# Compute the fault distance of a memory circuit

A syndrome-extraction circuit can spread an ancilla fault to several data qubits, making the interaction schedule important to the protection a code provides.
To assess that protection, we compute a circuit's fault distance: the minimum number of modeled error mechanisms that can flip a tracked logical observable without triggering a detector.
A detector records a change in measurement parity that would be predictable without noise, while a logical observable tracks the encoded information.
We compare a Steane-code memory circuit with a bivariate bicycle memory circuit to see when the extraction circuit preserves the code distance.

The [memory benchmarks](2_quantum_memory_x_or_z.ipynb) estimate failure probabilities from samples.
Here an active-operation depolarizing noise model specifies possible faults in gates, resets, and measurements, and an exact search finds the smallest undetected logical fault.
The result is a structural property of that model and circuit, independent of the decoder used in a sampling benchmark.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import stim
import sympy

from qldpc import circuits, codes

## A small extraction circuit that loses distance

The Steane code has distance 3, but the bare-ancilla extraction circuit below has fault distance 2 in each measured logical basis.
Here bare-ancilla extraction measures stabilizers without additional flag ancillas to detect fault propagation.
Initialization and final readout are ideal; only one syndrome-extraction round receives active-operation depolarizing noise, with idle noise disabled.
Thus this diagnostic isolates the extraction circuit rather than testing noisy boundaries or a long memory experiment.

In [3]:
noise_model = circuits.DepolarizingNoiseModel(1e-3, include_idling_error=False)


def get_noisy_memory_circuit(code: codes.CSSCode, basis: str, num_rounds: int = 1) -> stim.Circuit:
    """Keep initialization and readout ideal while adding noise to the QEC cycle."""
    parts = circuits.get_memory_experiment_parts(code, basis=basis, num_rounds=num_rounds)
    return parts.initialization + noise_model.noisy_circuit(parts.qec_cycle) + parts.readout


code = codes.SteaneCode()
print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(f"{basis}-basis memory: fault distance {fault_distance}")

code parameters: (7, 1, 3)


X-basis memory: fault distance 2
Z-basis memory: fault distance 2


Distance 2 means that a pair of modeled mechanisms can evade the detectors while flipping a tracked logical observable.
This example shows how an extraction circuit can have a smaller fault distance than the underlying code.
Flagged, Shor, or Steane-type extraction offers alternative circuits to investigate; [Steane-swap QEC](6_steane_qec.ipynb) develops one such workflow.

## What the exact search computes

An error mechanism is an error instruction in the circuit's detector error model (DEM).
Mechanisms with identical detector and observable effects can be merged, so mechanism indices are not physical gate-location indices.
The fault weight is the number of mechanisms in a combination.
`circuits.get_fault_distance` searches for a minimum-weight combination with no detector flips and at least one observable flip.
It converts that problem to the Z-distance of an auxiliary Calderbank–Shor–Steane (CSS) code, with pure-X and pure-Z check generators, and uses the Brouwer–Zimmermann distance algorithm.
The [API derivation](https://qldpc.readthedocs.io/autoapi/qldpc/circuits/certification/fault_distance/index.html) describes this reduction.

The search is exact but can be expensive: its cost can grow exponentially with fault distance.
The nonzero noise parameter fixes the modeled mechanisms; this minimum-weight question does not use their probabilities to rank faults.

## A bivariate bicycle extraction circuit

For comparison, we use a bivariate bicycle code from [arXiv:2308.07915](https://arxiv.org/abs/2308.07915) with parameters $[[72,12,6]]$: 72 physical qubits, 12 logical qubits, and distance 6.
The memory circuit has the same ideal boundaries and one extraction round as the Steane example.
Here each basis has 432 mechanisms after equivalent ones are merged.
Checking all combinations of up to six would require roughly nine trillion combinations; the code-distance reduction avoids that exhaustive enumeration.

In [4]:
x, y = sympy.symbols("x y")
code = codes.BBCode({x: 6, y: 6}, x**3 + y + y**2, y**3 + x + x**2)

print(f"code parameters: {code.get_code_params()}")
for basis in ("X", "Z"):
    noisy_circuit = get_noisy_memory_circuit(code, basis)
    num_errors = noisy_circuit.detector_error_model().num_errors
    fault_distance = circuits.get_fault_distance(noisy_circuit)
    print(f"{basis}-basis memory: {num_errors} error mechanisms, fault distance {fault_distance}")

code parameters: (72, 12, 6)


X-basis memory: 432 error mechanisms, fault distance 6


Z-basis memory: 432 error mechanisms, fault distance 6


The resulting fault distance equals the supplied code distance in both bases for this circuit.
Together with the Steane result, this shows that preserving code distance depends on the code and extraction circuit, not only on choosing bare ancillas.
To compare schedules by sampled performance, see [AlphaSyndrome](specialized/alpha_syndrome.ipynb).

For preparation circuits, the [state-preparation notebook](5_state_preparation.ipynb) checks bounded fault propagation: whether accepted faults leave output errors whose residual weight is no greater than the number of fault mechanisms.